# Spark TRM on Colab (7–15M)

Phase 1–2 notebook: reproduce the tiny recursive model, then run one architecture probe.

**Before you run anything**

1. Runtime → Change runtime type → **GPU** (T4 is enough for `spark-7`; A100/L4 is nicer).
2. If `src/` is not on GitHub yet, zip the local `reasoning-model` folder and upload it in the setup cell, or put it on Drive.
3. Free Colab will disconnect. Mount Drive if you want checkpoints to survive.

Gate **G0** is still ≥ 40% ARC-AGI-1 public eval with `spark-7`. Do not jump to `spark-15` until that number exists.

In [ ]:
# @title Settings
REPO_URL = "https://github.com/arjun988/reasoning-model.git"  # @param {type:"string"}
CODE_SOURCE = "clone"  # @param ["clone", "drive", "upload_zip"]
DRIVE_DIR = "/content/drive/MyDrive/reasoning-model"  # @param {type:"string"}
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
RUN_MODE = "overfit"  # @param ["overfit", "spark7_smoke", "spark7_full", "probe_rope2d"]
BATCH_SIZE = 4  # @param {type:"integer"}
MAX_STEPS = 200  # @param {type:"integer"}

In [ ]:
import os, sys, subprocess, zipfile
from pathlib import Path

def sh(cmd):
    print("+", cmd)
    subprocess.check_call(cmd, shell=True)

import torch
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
if not torch.cuda.is_available():
    print("WARNING: no GPU. Runtime → Change runtime type → GPU, then rerun from here.")

In [ ]:
from google.colab import drive, files

ROOT = Path("/content/reasoning-model")

if SAVE_TO_DRIVE or CODE_SOURCE == "drive":
    drive.mount("/content/drive")

if CODE_SOURCE == "clone":
    if not ROOT.exists():
        sh(f"git clone --depth 1 {REPO_URL} {ROOT}")
    else:
        print("already cloned", ROOT)
elif CODE_SOURCE == "drive":
    src = Path(DRIVE_DIR)
    if not src.exists():
        raise FileNotFoundError(f"Put the repo at {src} on Drive, or switch CODE_SOURCE to upload_zip")
    if ROOT.exists() or ROOT.is_symlink():
        print("using", ROOT)
    else:
        ROOT.symlink_to(src, target_is_directory=True)
        print("linked", ROOT, "->", src)
elif CODE_SOURCE == "upload_zip":
    print("Upload reasoning-model.zip (the folder that contains src/ and configs/)")
    uploaded = files.upload()
    zpath = next(iter(uploaded))
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall("/content/_upload")
    hits = list(Path("/content/_upload").rglob("src/spark/recursive.py"))
    if not hits:
        raise FileNotFoundError("zip did not contain src/spark/recursive.py")
    ROOT = hits[0].parents[2]
    print("unpacked at", ROOT)
else:
    raise ValueError(CODE_SOURCE)

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("cwd", os.getcwd())
print("has src", (ROOT / "src" / "spark" / "recursive.py").exists())
if not (ROOT / "src" / "spark" / "recursive.py").exists():
    raise FileNotFoundError(
        "GitHub clone has no training code yet. Re-run with CODE_SOURCE='upload_zip' "
        "or 'drive' after you copy the local repo."
    )

In [ ]:
# Colab already ships a CUDA torch. Only install the extra project deps.
sh("pip -q install pyyaml pillow pytest")
sh("python scripts/count_params.py configs/spark7.yaml configs/spark15.yaml configs/spark_debug.yaml")

In [ ]:
sh("python scripts/download_arc.py")
from src.arc.io import load_arc_dir
train_tasks = load_arc_dir("data/arc-agi/data/training", split="training")
eval_tasks = load_arc_dir("data/arc-agi/data/evaluation", split="evaluation")
print(f"ARC-AGI-1 train={len(train_tasks)} eval={len(eval_tasks)}")

In [ ]:
from src.spark.config import merge_configs
from src.train.train_arc import train

device = "cuda" if torch.cuda.is_available() else "cpu"
print("RUN_MODE", RUN_MODE, "device", device)

if RUN_MODE == "overfit":
    arch, cfg = merge_configs("configs/spark_debug.yaml", "configs/train_arc.yaml")
    cfg.max_steps = MAX_STEPS
    cfg.batch_size = BATCH_SIZE
    cfg.device = device
    cfg.out_dir = "artifacts/overfit"
    cfg.eval_every = 50
    cfg.save_every = 200
    cfg.log_every = 10
    cfg.warmup_steps = 20
    train(arch, cfg, synthetic=True)
elif RUN_MODE == "spark7_smoke":
    arch, cfg = merge_configs("configs/spark7.yaml", "configs/train_arc.yaml")
    cfg.max_steps = MAX_STEPS
    cfg.batch_size = BATCH_SIZE
    cfg.device = device
    cfg.data_dir = "data/arc-agi"
    cfg.out_dir = "artifacts/spark7_smoke"
    cfg.eval_every = max(50, MAX_STEPS // 4)
    cfg.save_every = max(100, MAX_STEPS // 2)
    cfg.log_every = 10
    cfg.warmup_steps = min(200, MAX_STEPS // 5)
    train(arch, cfg, synthetic=False)
elif RUN_MODE == "spark7_full":
    arch, cfg = merge_configs("configs/spark7.yaml", "configs/train_arc.yaml")
    cfg.batch_size = BATCH_SIZE
    cfg.device = device
    cfg.data_dir = "data/arc-agi"
    cfg.out_dir = "artifacts/spark7"
    train(arch, cfg, synthetic=False)
elif RUN_MODE == "probe_rope2d":
    arch, cfg = merge_configs("configs/probes/rope_2d.yaml", "configs/train_arc.yaml")
    cfg.max_steps = MAX_STEPS
    cfg.batch_size = BATCH_SIZE
    cfg.device = device
    cfg.data_dir = "data/arc-agi"
    cfg.out_dir = "artifacts/probe_rope2d"
    train(arch, cfg, synthetic=False)
else:
    raise ValueError(RUN_MODE)

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

log_candidates = sorted(Path("artifacts").rglob("train.jsonl"))
print("logs:", [str(p) for p in log_candidates])
if log_candidates:
    rows = [json.loads(l) for l in log_candidates[-1].read_text().splitlines() if l.strip()]
    train_rows = [r for r in rows if "split" not in r and "loss" in r]
    eval_rows = [r for r in rows if r.get("split") == "eval"]
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    if train_rows:
        ax[0].plot([r["step"] for r in train_rows], [r["loss"] for r in train_rows], label="train loss")
        ax[0].plot([r["step"] for r in train_rows], [r.get("exact", 0) for r in train_rows], label="train exact")
        ax[0].legend(); ax[0].set_title("train"); ax[0].set_xlabel("step")
    if eval_rows:
        ax[1].plot([r["step"] for r in eval_rows], [r["exact"] for r in eval_rows], label="eval exact")
        ax[1].plot([r["step"] for r in eval_rows], [r["cell_acc"] for r in eval_rows], label="eval cell")
        ax[1].legend(); ax[1].set_title("eval"); ax[1].set_xlabel("step")
    plt.tight_layout()
    plt.show()
    print("last train", train_rows[-1] if train_rows else None)
    print("last eval", eval_rows[-1] if eval_rows else None)

In [ ]:
import shutil

if SAVE_TO_DRIVE:
    dest = Path(DRIVE_DIR) / "artifacts"
    dest.mkdir(parents=True, exist_ok=True)
    src = Path("artifacts")
    if src.exists():
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print("copied artifacts ->", dest)
    else:
        print("no artifacts/ to copy")

## Suggested Colab sequence

| Pass | `RUN_MODE` | `MAX_STEPS` | Time (T4) |
|---|---|---|---|
| 1 | `overfit` | 200 | a few minutes |
| 2 | `spark7_smoke` | 200 | ~15–40 min |
| 3 | `spark7_full` | leave default (50k) | many hours / resume from Drive |
| 4 | `probe_rope2d` | after G0 only | same as full |

If the clone step says the training code is missing, switch `CODE_SOURCE` to `upload_zip` and upload a zip of this repo (must contain `src/` and `configs/`).

T4 15 GB: keep `BATCH_SIZE` at 4 (or 2 if you OOM). Do not start `spark-15` on free Colab until `spark-7` is stable.